<!-- codex-architecture-notes -->
## Architectural Notes

**Purpose:** Preprocesses the raw ACD degree-course table into the cleaned curriculum metadata table.

**Notebook Shape:** 16 cells (16 code, 0 markdown).

**Inputs / Data Sources:**
- `df = pd.read_parquet(raw_path)`

**Outputs / Side Effects:**
- `df1.to_parquet(r"D:\AI\Real projects\Academic_Advisor\data\preprocessed\V_ACD_DEGREE_COURSE\v_acd_degree_course.parquet", index=False)`

**Logic Flow:**
1. Load raw degree-course parquet.
2. Normalize IDs and selected fields.
3. Inspect data quality.
4. Write cleaned ACD degree-course parquet.

**Maintainability Notes:** Curriculum metadata drives requirement and difficulty features; validate duplicate degree-course keys before downstream joins.


In [17]:
import pandas as pd

from src.cleaning_utils import normalize_id_columns
from src.paths import RAW_DIR, CLEAN_DIR
from src.io_utils import save_parquet

raw_path = RAW_DIR / "v_acd_degree_course.parquet"


df = pd.read_parquet(raw_path)


In [18]:
df

,degree_course_id,course_id,degree_id,faculty_id,course_type_id,requirement_type_id,requirement_type_sl,course_name_sl,course_official_sl,degree_name_sl,year_order,semester_order,required_credits,course_credits,active,credits_count,req_degree_id
0,1.111,917.111,17.111,NaN,3.0,5.0,متطلبات الشهادة الإجبارية,الكيمياء الحيوية,الكيمياء الحيوية,الصيدلة,3.0,1.0,NaN,4.0,A,193.0,17.111
1,2.111,923.111,17.111,NaN,3.0,5.0,متطلبات الشهادة الإجبارية,الكيمياء الحيوية التطبيقية,الكيمياء الحيوية التطبيقية,الصيدلة,3.0,2.0,NaN,3.0,A,193.0,17.111
2,3.111,908.111,17.111,NaN,1.0,5.0,متطلبات الشهادة الإجبارية,الإحصاء الحيوي,الإحصاء الحيوي,الصيدلة,2.0,1.0,NaN,2.0,A,193.0,17.111
3,4.111,891.111,17.111,NaN,3.0,5.0,متطلبات الشهادة الإجبارية,البيولوجيا النباتية,البيولوجيا النباتية,الصيدلة,1.0,1.0,NaN,4.0,A,193.0,17.111
4,5.111,907.111,17.111,NaN,1.0,5.0,متطلبات الشهادة الإجبارية,تجارة الأدوية,تجارة الأدوية,الصيدلة,2.0,1.0,NaN,2.0,A,193.0,17.111
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4001,4135.111,1664.111,65.111,195.111,3.0,3.0,متطلبات الكلية الإجبارية,اقتصاديات البناء والتشييد,اقتصاديات البناء والتشييد,هندسة تكنولوجيا البناء والتشييد,5.0,2.0,140.0,3.0,A,165.0,NaN
4002,4136.111,1665.111,65.111,195.111,1.0,3.0,متطلبات الكلية الإجبارية,إدارة الجودة في التشييد والبناء,إدارة الجودة في التشييد والبناء,هندسة تكنولوجيا البناء والتشييد,5.0,2.0,NaN,2.0,A,165.0,NaN
4003,4137.111,1666.111,65.111,195.111,1.0,3.0,متطلبات الكلية الإجبارية,آداب واخلاقيات العمل الهندسي,آداب واخلاقيات العمل الهندسي,هندسة تكنولوجيا البناء والتشييد,5.0,2.0,140.0,2.0,A,165.0,NaN
4004,4138.111,1667.111,65.111,195.111,2.0,3.0,متطلبات الكلية الإجبارية,التدريب العملي الميداني,التدريب العملي الميداني,هندسة تكنولوجيا البناء والتشييد,5.0,2.0,140.0,3.0,A,165.0,NaN


In [19]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4006 entries, 0 to 4005
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   degree_course_id     4006 non-null   float64
 1   course_id            4006 non-null   float64
 2   degree_id            4006 non-null   float64
 3   faculty_id           1526 non-null   float64
 4   course_type_id       4006 non-null   float64
 5   requirement_type_id  4006 non-null   float64
 6   requirement_type_sl  4006 non-null   str    
 7   course_name_sl       4006 non-null   str    
 8   course_official_sl   4006 non-null   str    
 9   degree_name_sl       4006 non-null   str    
 10  year_order           3997 non-null   float64
 11  semester_order       3997 non-null   float64
 12  required_credits     296 non-null    float64
 13  course_credits       4006 non-null   float64
 14  active               4006 non-null   str    
 15  credits_count        4006 non-null   float64
 16 

In [20]:
df=df.drop(columns=['faculty_id','required_credits','req_degree_id','course_official_sl','course_type_id','year_order','semester_order','active'])

In [21]:
df.head()

,degree_course_id,course_id,degree_id,requirement_type_id,requirement_type_sl,course_name_sl,degree_name_sl,course_credits,credits_count
0,1.111,917.111,17.111,5.0,متطلبات الشهادة الإجبارية,الكيمياء الحيوية,الصيدلة,4.0,193.0
1,2.111,923.111,17.111,5.0,متطلبات الشهادة الإجبارية,الكيمياء الحيوية التطبيقية,الصيدلة,3.0,193.0
2,3.111,908.111,17.111,5.0,متطلبات الشهادة الإجبارية,الإحصاء الحيوي,الصيدلة,2.0,193.0
3,4.111,891.111,17.111,5.0,متطلبات الشهادة الإجبارية,البيولوجيا النباتية,الصيدلة,4.0,193.0
4,5.111,907.111,17.111,5.0,متطلبات الشهادة الإجبارية,تجارة الأدوية,الصيدلة,2.0,193.0


In [22]:
df['requirement_type_id'].value_counts()

requirement_type_id
5.0    1407
3.0    1259
2.0     531
1.0     350
4.0     267
6.0     192
Name: count, dtype: int64

In [23]:
df['course_credits'].eq(24).value_counts()

course_credits
False    4005
True        1
Name: count, dtype: int64

In [24]:
df['course_credits'].value_counts()

course_credits
3.0     2385
2.0     1144
4.0      229
0.0      108
1.0       71
5.0       36
6.0       24
4.5        8
24.0       1
Name: count, dtype: int64

In [25]:
df.drop_duplicates()

,degree_course_id,course_id,degree_id,requirement_type_id,requirement_type_sl,course_name_sl,degree_name_sl,course_credits,credits_count
0,1.111,917.111,17.111,5.0,متطلبات الشهادة الإجبارية,الكيمياء الحيوية,الصيدلة,4.0,193.0
1,2.111,923.111,17.111,5.0,متطلبات الشهادة الإجبارية,الكيمياء الحيوية التطبيقية,الصيدلة,3.0,193.0
2,3.111,908.111,17.111,5.0,متطلبات الشهادة الإجبارية,الإحصاء الحيوي,الصيدلة,2.0,193.0
3,4.111,891.111,17.111,5.0,متطلبات الشهادة الإجبارية,البيولوجيا النباتية,الصيدلة,4.0,193.0
4,5.111,907.111,17.111,5.0,متطلبات الشهادة الإجبارية,تجارة الأدوية,الصيدلة,2.0,193.0
...,...,...,...,...,...,...,...,...,...
4001,4135.111,1664.111,65.111,3.0,متطلبات الكلية الإجبارية,اقتصاديات البناء والتشييد,هندسة تكنولوجيا البناء والتشييد,3.0,165.0
4002,4136.111,1665.111,65.111,3.0,متطلبات الكلية الإجبارية,إدارة الجودة في التشييد والبناء,هندسة تكنولوجيا البناء والتشييد,2.0,165.0
4003,4137.111,1666.111,65.111,3.0,متطلبات الكلية الإجبارية,آداب واخلاقيات العمل الهندسي,هندسة تكنولوجيا البناء والتشييد,2.0,165.0
4004,4138.111,1667.111,65.111,3.0,متطلبات الكلية الإجبارية,التدريب العملي الميداني,هندسة تكنولوجيا البناء والتشييد,3.0,165.0


In [26]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4006 entries, 0 to 4005
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   degree_course_id     4006 non-null   float64
 1   course_id            4006 non-null   float64
 2   degree_id            4006 non-null   float64
 3   requirement_type_id  4006 non-null   float64
 4   requirement_type_sl  4006 non-null   str    
 5   course_name_sl       4006 non-null   str    
 6   degree_name_sl       4006 non-null   str    
 7   course_credits       4006 non-null   float64
 8   credits_count        4006 non-null   float64
dtypes: float64(6), str(3)
memory usage: 820.9 KB


In [27]:
df1 = normalize_id_columns(
    df,
    column_map=[
        ("degree_course_id", "degree_course_id_key"),
        ("course_id", "course_id_key"),
        ("degree_id", "degree_id_key"),
    ],
)

df1["requirement_type_id"] = pd.to_numeric(
    df1["requirement_type_id"],
    errors="coerce"
).astype("Int64")
df1["credits_count"] = pd.to_numeric(
    df1["credits_count"],
    errors="coerce"
).astype("Int64")

In [28]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 4006 entries, 0 to 4005
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   degree_course_id      4006 non-null   float64
 1   course_id             4006 non-null   float64
 2   degree_id             4006 non-null   float64
 3   requirement_type_id   4006 non-null   Int64  
 4   requirement_type_sl   4006 non-null   str    
 5   course_name_sl        4006 non-null   str    
 6   degree_name_sl        4006 non-null   str    
 7   course_credits        4006 non-null   float64
 8   credits_count         4006 non-null   Int64  
 9   degree_course_id_key  4006 non-null   string 
 10  course_id_key         4006 non-null   string 
 11  degree_id_key         4006 non-null   string 
dtypes: Int64(2), float64(4), str(3), string(3)
memory usage: 1005.3 KB


In [29]:
df1.head()

,degree_course_id,course_id,degree_id,requirement_type_id,requirement_type_sl,course_name_sl,degree_name_sl,course_credits,credits_count,degree_course_id_key,course_id_key,degree_id_key
0,1.111,917.111,17.111,5,متطلبات الشهادة الإجبارية,الكيمياء الحيوية,الصيدلة,4.0,193,1.111,917.111,17.111
1,2.111,923.111,17.111,5,متطلبات الشهادة الإجبارية,الكيمياء الحيوية التطبيقية,الصيدلة,3.0,193,2.111,923.111,17.111
2,3.111,908.111,17.111,5,متطلبات الشهادة الإجبارية,الإحصاء الحيوي,الصيدلة,2.0,193,3.111,908.111,17.111
3,4.111,891.111,17.111,5,متطلبات الشهادة الإجبارية,البيولوجيا النباتية,الصيدلة,4.0,193,4.111,891.111,17.111
4,5.111,907.111,17.111,5,متطلبات الشهادة الإجبارية,تجارة الأدوية,الصيدلة,2.0,193,5.111,907.111,17.111


In [30]:
a=df1['credits_count'].value_counts()

len(a)


32

In [31]:
save_parquet(df1, CLEAN_DIR / "V_ACD_DEGREE_COURSE" / "clean_v_acd_degree_course.parquet")

WindowsPath('D:/AI/Real projects/Academic_Advisor/data/preprocessed/V_ACD_DEGREE_COURSE/clean_v_acd_degree_course.parquet')